## L1: Automated Project: Planning, Estimation, and Allocation

### Environment Setup & Imports
Suppress warnings, load API keys, and import required libraries from CrewAI and LangChain.

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

import os
# Load environment variables
from helper import load_env
load_env()
from helper import get_openrouter_api_key
get_openrouter_api_key()

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
os.environ["OPENROUTER_API_KEY"] = openrouter_api_key
os.environ["OPENROUTER_API_BASE"] = "https://openrouter.ai/api/v1"
os.environ["OPENROUTER_MODEL_NAME"] = 'mistralai/mistral-7b-instruct'


import yaml
from crewai import Agent, Task, Crew
from langchain_openai import ChatOpenAI
from crewai_tools import DirectoryReadTool, \
                         FileReadTool, \
                         SerperDevTool

### Load LLM and Configurations
Initialize the ChatOpenAI model using OpenRouter and load agent/task configurations from YAML files.

In [ ]:
openrouter_llm = ChatOpenAI(
    model=os.environ["OPENROUTER_API_KEY"],
    openai_api_base=os.environ["OPENROUTER_API_BASE"],
    openai_api_key=os.environ["OPENROUTER_API_KEY"]
)


# Define file paths for YAML configurations
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
print(configs)
agents_config = configs['agents']
tasks_config = configs['tasks']

### Define Output Data Structures
Create Pydantic models (`TaskEstimate`, `Milestone`, `ProjectPlan`) to ensure the Crew returns well-structured, predictable data.

In [ ]:
from typing import List
from pydantic import BaseModel, Field

class TaskEstimate(BaseModel):
    task_name: str = Field(..., description="Name of the task")
    estimated_time_hours: float = Field(..., description="Estimated time to complete the task in hours")
    required_resources: List[str] = Field(..., description="List of resources required to complete the task")

class Milestone(BaseModel):
    milestone_name: str = Field(..., description="Name of the milestone")
    tasks: List[str] = Field(..., description="List of task IDs associated with this milestone")

class ProjectPlan(BaseModel):
    tasks: List[TaskEstimate] = Field(..., description="List of tasks with their estimates")
    milestones: List[Milestone] = Field(..., description="List of project milestones")

### Instantiate Agents, Tasks, and Crew
Create individual agents and tasks using the loaded YAML configurations, and assemble them into a Crew for execution.

In [ ]:
# Creating Agents
project_planning_agent = Agent(
  config=agents_config['project_planning_agent']
)

estimation_agent = Agent(
  config=agents_config['estimation_agent']
)

resource_allocation_agent = Agent(
  config=agents_config['resource_allocation_agent']
)

# Creating Tasks
task_breakdown = Task(
  config=tasks_config['task_breakdown'],
  agent=project_planning_agent
)

time_resource_estimation = Task(
  config=tasks_config['time_resource_estimation'],
  agent=estimation_agent
)

resource_allocation = Task(
  config=tasks_config['resource_allocation'],
  agent=resource_allocation_agent,
  output_pydantic=ProjectPlan # This is the structured output we want
)

# Creating Crew
crew = Crew(
  agents=[
    project_planning_agent,
    estimation_agent,
    resource_allocation_agent
  ],
  tasks=[
    task_breakdown,
    time_resource_estimation,
    resource_allocation
  ],
  verbose=True
)

### Define Project Details
Set up specific details about the project, such as objectives, team members, and requirements, and display them in the notebook.

In [ ]:
from IPython.display import display, Markdown

project = 'Website'
industry = 'Technology'
project_objectives = 'Create a website for a small business'
team_members = """
- John Doe (Project Manager)
- Jane Doe (Software Engineer)
- Bob Smith (Designer)
- Alice Johnson (QA Engineer)
- Tom Brown (QA Engineer)
"""
project_requirements = """
- Create a responsive design that works well on desktop and mobile devices
- Implement a modern, visually appealing user interface with a clean look
- Develop a user-friendly navigation system with intuitive menu structure
- Include an "About Us" page highlighting the company's history and values
- Design a "Services" page showcasing the business's offerings with descriptions
- Create a "Contact Us" page with a form and integrated map for communication
- Implement a blog section for sharing industry news and company updates
- Ensure fast loading times and optimize for search engines (SEO)
- Integrate social media links and sharing capabilities
- Include a testimonials section to showcase customer feedback and build trust
"""

# Format the dictionary as Markdown for a better display in Jupyter Lab
formatted_output = f"""
**Project Type:** {project}

**Project Objectives:** {project_objectives}

**Industry:** {industry}

**Team Members:**
{team_members}
**Project Requirements:**
{project_requirements}
"""
# Display the formatted output as Markdown
display(Markdown(formatted_output))

### Execute the Crew
Pass the project details as inputs and kick off the CrewAI workflow to generate the project plan.

In [ ]:
# The given Python dictionary
inputs = {
  'project_type': project,
  'project_objectives': project_objectives,
  'industry': industry,
  'team_members': team_members,
  'project_requirements': project_requirements
}

# Run the crew
result = crew.kickoff(
  inputs=inputs
)

### Analyze Usage Metrics and Costs
Calculate the total API cost based on the number of prompt and completion tokens used during the execution.

In [ ]:
import pandas as pd

costs = 0.150 * (crew.usage_metrics.prompt_tokens + crew.usage_metrics.completion_tokens) / 1_000_000
print(f"Total costs: ${costs:.4f}")

# Convert UsageMetrics instance to a DataFrame
df_usage_metrics = pd.DataFrame([crew.usage_metrics.dict()])
df_usage_metrics

### Extract Structured Output
Convert the final structured output from a Pydantic object back into a standard Python dictionary for easier manipulation.

In [ ]:
result.pydantic.dict()

### Display Task Breakdown
Extract the generated tasks from the result and display them as a formatted HTML table using Pandas.

In [ ]:
tasks = result.pydantic.dict()['tasks']
df_tasks = pd.DataFrame(tasks)

# Display the DataFrame as an HTML table
df_tasks.style.set_table_attributes('border="1"').set_caption("Task Details").set_table_styles(
    [{'selector': 'th, td', 'props': [('font-size', '120%')]}]
)

### Display Project Milestones
Extract the generated milestones and display them as a formatted HTML table using Pandas.

In [ ]:
milestones = result.pydantic.dict()['milestones']
df_milestones = pd.DataFrame(milestones)

# Display the DataFrame as an HTML table
df_milestones.style.set_table_attributes('border="1"').set_caption("Task Details").set_table_styles(
    [{'selector': 'th, td', 'props': [('font-size', '120%')]}]
)